# Civil Litigation Outcome Prediction Model

## Complete Codebase · Schema · Neural Network · Weighted Prediction Engine

### Consumer Banking Litigation · FJC Dataset · Binary Classification

This notebook is **self-contained single-file** containing entire prediction system:

1. **Schema & Constants** — All FJC variable definitions, NOS codes, circuit codes, origin codes, disposition mapping, doctrinal eras
2. **Real FJC Dataset** — 10,000+ resolved federal cases (2020–2025) loaded and processed
3. **TensorFlow/Keras Neural Network** — Binary classifier (32→128→64→32→1 sigmoid) trained on real data
4. **Weighted Prediction Engine** — Full Python port of the web app's TypeScript engine with seminal case analysis
5. **Interactive Prediction** — Run predictions with both models and compare
6. **Evaluation & Metrics** — AUC, precision, recall, confusion matrix, per-feature analysis

---

### 8 FJC Predictor Variables

| # | Variable | FJC Field | Encoding |
|---|----------|-----------|----------|
| 1 | NOS Code | NOS | One-hot (4 codes: 480, 190, 371, 370) |
| 2 | Origin Code | ORIGIN | One-hot (8 categories) |
| 3 | Class Action | CLASSACT | Binary (-8 = no, 1 = yes) |
| 4 | Circuit Code | CIRCUIT | One-hot (12 circuits: 0=DC, 1-11) |
| 5 | Year Filed | FILEDATE | Normalized continuous |
| 6 | Arbitration | ARBIT | Binary (-8 = no, E/V/M/Y = yes) |
| 7 | MDL Status | MDLDOCK | Binary (-8 = no, numeric = yes) |
| 8 | Doctrinal Era | Derived from Year | One-hot (4 eras) |

### Binary Outcome (Target)

| Label | FJC DISP Codes | Description |
|-------|---------------|-------------|
| 1 (Plaintiff-favorable) | 1, 6, 7 | Judgment for Plaintiff + Settled + Judgment on Consent |
| 0 (Not plaintiff-favorable) | All others | Dismissal, defendant judgment, procedural, transfer |

### Doctrinal Eras (Seminal Cases)

| Era | Period | Key Case |
|-----|--------|----------|
| Pre-Concepcion | Before 2011 | AT&T v. Concepcion |
| Post-Concepcion | 2011–2016 | Arbitration enforcement |
| Post-Spokeo | 2016–2021 | Spokeo v. Robins (standing) |
| Post-TransUnion | 2021–present | TransUnion v. Ramirez |

### Supplemental Variables (Weighted Engine Only)

| Variable | Source |
|----------|--------|
| Concrete Injury Documented | Spokeo/TransUnion standing analysis |
| Prior Regulatory Action | CashCall/CFPB enforcement |
| Defendant Bank | Institutional risk profile |

---
## Part 1: Dependencies & Environment

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
import numpy as np
import pandas as pd
import json
import uuid
from datetime import datetime
from typing import Dict, List, Tuple, Optional


import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## Part 2: Complete Schema — FJC Variable Definitions & Constants

This section is the Python equivalent of `shared/schema.ts` from the web app codebase.
Every constant, code mapping, and type definition used across the system is defined here.

In [ ]:
NOS_CODE_LABELS = {
    '110': 'Insurance', '120': 'Marine', '130': 'Miller Act', '140': 'Negotiable Instrument',
    '150': 'Recovery of Overpayment', '151': 'Medicare Act', '152': 'Recovery of Student Loans',
    '153': 'Recovery of Veteran Benefits', '160': "Stockholders' Suits", '190': 'Other Contract',
    '195': 'Contract Product Liability', '196': 'Franchise', '210': 'Land Condemnation',
    '220': 'Foreclosure', '230': 'Rent Lease & Ejectment', '240': 'Torts to Land',
    '245': 'Tort Product Liability', '290': 'All Other Real Property', '310': 'Airplane',
    '315': 'Airplane Product Liability', '320': 'Assault, Libel & Slander',
    '330': "Fed Employers' Liability", '340': 'Marine', '345': 'Marine Product Liability',
    '350': 'Motor Vehicle', '355': 'Motor Vehicle Product Liability',
    '360': 'Other Personal Injury', '362': 'Personal Injury - Medical Malpractice',
    '365': 'Personal Injury - Product Liability', '367': 'Health Care/Pharmaceutical',
    '368': 'Asbestos Personal Injury', '370': 'Other Fraud', '371': 'Truth in Lending',
    '375': 'False Claims Act', '376': 'Qui Tam (31 USC 3729a)',
    '380': 'Other Personal Property Damage', '385': 'Property Damage Product Liability',
    '400': 'State Reapportionment', '410': 'Antitrust',
    '422': 'Bankruptcy Appeals (28 USC 158)', '423': 'Withdrawal 28 USC 157',
    '430': 'Banks and Banking', '440': 'Other Civil Rights', '441': 'Voting',
    '442': 'Employment', '443': 'Housing/Accommodations', '444': 'Welfare',
    '445': 'Americans with Disabilities (Employment)',
    '446': 'Americans with Disabilities (Other)', '448': 'Education', '450': 'Commerce',
    '460': 'Deportation', '462': 'Naturalization Application',
    '463': 'Habeas Corpus - Alien Detainee', '465': 'Other Immigration Actions',
    '470': 'RICO', '480': 'Consumer Credit', '490': 'Cable/Sat TV',
    '510': 'Motions to Vacate Sentence', '530': 'General', '535': 'Death Penalty',
    '540': 'Mandamus & Other', '550': 'Civil Rights', '555': 'Prison Condition',
    '560': 'Civil Detainee', '610': 'Agricultural Acts', '620': 'Other Food & Drug',
    '625': 'Drug Related Seizure of Property', '630': 'Liquor Laws',
    '640': 'Railroad & Trucks', '650': 'Airline Regulations',
    '660': 'Occupational Safety/Health', '690': 'Other',
    '710': 'Fair Labor Standards Act', '720': 'Labor/Management Relations',
    '730': 'Labor/Management Reporting & Disclosure Act', '740': 'Railway Labor Act',
    '751': 'Family and Medical Leave Act', '790': 'Other Labor Litigation',
    '791': 'ERISA', '810': 'Selective Service', '820': 'Copyrights', '830': 'Patent',
    '835': 'Patent - Abbreviated New Drug Application', '840': 'Trademark',
    '850': 'Securities/Commodities/Exchange', '860': 'Social Security',
    '861': 'HIA (1395ff)', '862': 'Black Lung (923)', '863': 'DIWC/DIWW (405g)',
    '864': 'SSID (Title XVI)', '865': 'RSI (405g)',
    '870': 'Taxes (US Plaintiff or Defendant)', '871': 'IRS - Third Party',
    '875': 'Customer Challenge 12 USC 3410', '890': 'Other Statutory Actions',
    '891': 'Agricultural Acts', '893': 'Environmental Matters',
    '895': 'Freedom of Information Act', '896': 'Arbitration',
    '899': 'Administrative Procedure Act/Review or Appeal of Agency Decision',
    '900': 'Appeal of Fee Determination Under Equal Access to Justice',
    '910': 'Domestic Relations', '920': 'Insanity', '930': 'Probate',
    '940': 'Substitute Trustee', '950': 'Constitutionality of State Statutes',
}

CONSUMER_BANKING_NOS_CODES = ['480', '190', '371', '370']

NOS_CODE_DESCRIPTIONS = {
    '480': 'Consumer Credit (FCRA, FDCPA, EFTA, ECOA)',
    '190': 'Banks and Banking (Other Contract)',
    '371': 'Truth in Lending (TILA/Reg Z)',
    '370': 'Other Fraud (UDAAP, deceptive practices)',
}

ORIGIN_CODES = ['1', '2', '3', '4', '5', '6', '7', '8']

ORIGIN_CODE_LABELS = {
    '1': 'Original Proceeding',
    '2': 'Removed from State Court',
    '3': 'Remanded from Appellate Court',
    '4': 'Reinstated or Reopened',
    '5': 'Transferred from Another District',
    '6': 'Multidistrict Litigation (MDL)',
    '7': 'Appeal to District Judge of Magistrate Judge Decision',
    '8': 'Multi-district Litigation (Originating)',
}

CIRCUIT_CODES = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']

CIRCUIT_CODE_LABELS = {
    '0': 'D.C. Circuit', '1': '1st Circuit', '2': '2nd Circuit', '3': '3rd Circuit',
    '4': '4th Circuit', '5': '5th Circuit', '6': '6th Circuit', '7': '7th Circuit',
    '8': '8th Circuit', '9': '9th Circuit', '10': '10th Circuit', '11': '11th Circuit',
    'DC': 'D.C. Circuit',
}

DISPOSITION_CODE_LABELS = {
    '0': 'Transfer to another district', '1': 'Remanded to state court',
    '2': 'Dismissed — want of prosecution', '3': 'Dismissed — lack of jurisdiction',
    '4': 'Judgment on default', '5': 'Judgment on consent',
    '6': 'Judgment on motion before trial', '7': 'Judgment on jury verdict',
    '8': 'Judgment on directed verdict', '9': 'Judgment on court trial',
    '10': 'Multi-district litigation transfer', '11': 'Remanded to U.S. Agency',
    '12': 'Voluntarily dismissed', '13': 'Settled', '14': 'Other',
    '15': 'Judgment for plaintiff (statistical closing)',
    '16': 'Judgment for defendant (statistical closing)',
    '17': 'Dismissed for want of prosecution (statistical closing)',
    '18': 'Dismissed — other (statistical closing)',
    '19': 'Transfer (statistical closing)', '20': 'Consolidation (statistical closing)',
}

FJC_DISPOSITION_MAP = {'1': 'plaintiff-favorable', '6': 'plaintiff-favorable', '7': 'plaintiff-favorable'}

PLAINTIFF_FAVORABLE_DISP = [1.0, 6.0, 7.0]
PLAINTIFF_FAVORABLE_LABELS = {
    '1': 'Judgment for Plaintiff',
    '6': 'Dismissed — Settled',
    '7': 'Judgment on Consent',
}

DOCTRINAL_ERAS = ['pre-concepcion', 'post-concepcion-pre-spokeo', 'post-spokeo-pre-transunion', 'post-transunion']

DOCTRINAL_ERA_LABELS = {
    'pre-concepcion': 'Pre-Concepcion (before 2011)',
    'post-concepcion-pre-spokeo': 'Post-Concepcion / Pre-Spokeo (2011–2016)',
    'post-spokeo-pre-transunion': 'Post-Spokeo / Pre-TransUnion (2016–2021)',
    'post-transunion': 'Post-TransUnion (2021–present)',
}

ERA_NAMES = ['Pre-Concepcion', 'Post-Concepcion', 'Post-Spokeo', 'Post-TransUnion']

MAJOR_BANKS = [
    'JPMorgan Chase', 'Bank of America', 'Wells Fargo', 'Citibank', 'Capital One',
    'U.S. Bank', 'PNC Financial', 'TD Bank', 'Truist', 'Goldman Sachs',
    'Morgan Stanley', 'Ally Financial', 'Discover', 'Synchrony Financial',
    'CashCall / Western Sky', 'Other',
]

def get_doctrinal_era(year):
    if year < 2011: return 0
    if year < 2016: return 1
    if year < 2021: return 2
    return 3

def get_doctrinal_era_name(year):
    return DOCTRINAL_ERAS[get_doctrinal_era(year)]

def is_removed_from_state_court(origin):
    return str(origin) == '2'

def is_mdl_origin(origin):
    return str(origin) in ('6', '8')

N_NOS = len(CONSUMER_BANKING_NOS_CODES)   # 4
N_ORIGIN = len(ORIGIN_CODES)               # 8
N_CIRCUIT = len(CIRCUIT_CODES)             # 12
N_CLASS_ACTION = 1
N_ARBIT = 1
N_MDL = 1
N_YEAR = 1
N_ERA = 4

INPUT_DIM = N_NOS + N_ORIGIN + N_CIRCUIT + N_CLASS_ACTION + N_ARBIT + N_MDL + N_YEAR + N_ERA
OUTPUT_DIM = 1

print('=== SCHEMA LOADED ===')
print(f'NOS codes (consumer banking): {CONSUMER_BANKING_NOS_CODES}')
print(f'Origin codes: {ORIGIN_CODES}')
print(f'Circuit codes: {CIRCUIT_CODES}')
print(f'Plaintiff-favorable DISP: {PLAINTIFF_FAVORABLE_DISP}')
print(f'Doctrinal eras: {ERA_NAMES}')
print(f'Major banks: {len(MAJOR_BANKS)} institutions')
print(f'NN input dimension: {INPUT_DIM} (NOS={N_NOS} + ORIGIN={N_ORIGIN} + CIRCUIT={N_CIRCUIT} + CLASSACT={N_CLASS_ACTION} + ARBIT={N_ARBIT} + MDL={N_MDL} + YEAR={N_YEAR} + ERA={N_ERA})')
print(f'NN output: Binary sigmoid (plaintiff-favorable = 1, not = 0)')

### Schema Visual: Feature Encoding Pipeline

How raw FJC fields are transformed into the 32-dimensional neural network input vector.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('FJC Case → 32-Dim Feature Vector → Binary Prediction', fontsize=16, fontweight='bold', pad=20)

raw_fields = [
    ('NOS', '4 codes\n480, 190, 371, 370'),
    ('ORIGIN', '8 codes\n1–8'),
    ('CIRCUIT', '12 codes\n0(DC)–11'),
    ('CLASSACT', 'Binary\n-8 or 1'),
    ('ARBIT', 'Binary\n-8 or E/V/M/Y'),
    ('MDLDOCK', 'Binary\n-8 or numeric'),
    ('FILEDATE', 'Normalized\nyear'),
    ('ERA', '4 eras\none-hot'),
]

encodings = ['One-hot (4)', 'One-hot (8)', 'One-hot (12)', 'Binary (1)', 'Binary (1)', 'Binary (1)', 'Float (1)', 'One-hot (4)']
dims = [4, 8, 12, 1, 1, 1, 1, 4]
colors_raw = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c', '#e67e22', '#34495e']

for i, (field, desc) in enumerate(raw_fields):
    y = 10.5 - i * 1.2
    box = FancyBboxPatch((0.5, y - 0.35), 2.8, 0.7, boxstyle='round,pad=0.1',
                         facecolor=colors_raw[i], alpha=0.85, edgecolor='white', linewidth=1.5)
    ax.add_patch(box)
    ax.text(1.9, y, field, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    
    ax.annotate('', xy=(4.2, y), xytext=(3.4, y),
                arrowprops=dict(arrowstyle='->', color=colors_raw[i], lw=1.5))
    
    box2 = FancyBboxPatch((4.2, y - 0.35), 3.0, 0.7, boxstyle='round,pad=0.1',
                          facecolor=colors_raw[i], alpha=0.25, edgecolor=colors_raw[i], linewidth=1.5)
    ax.add_patch(box2)
    ax.text(5.7, y + 0.05, encodings[i], ha='center', va='center', fontsize=9, fontweight='bold')
    ax.text(5.7, y - 0.2, f'dim={dims[i]}', ha='center', va='center', fontsize=8, color='gray')

concat_y = 5.7
for i in range(len(raw_fields)):
    y = 10.5 - i * 1.2
    ax.annotate('', xy=(8.5, concat_y), xytext=(7.3, y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8, connectionstyle='arc3,rad=0.2'))

box_concat = FancyBboxPatch((8.5, concat_y - 0.5), 2.5, 1.0, boxstyle='round,pad=0.15',
                             facecolor='#2c3e50', alpha=0.9, edgecolor='white', linewidth=2)
ax.add_patch(box_concat)
ax.text(9.75, concat_y + 0.15, 'CONCAT', ha='center', va='center', fontsize=12, fontweight='bold', color='white')
ax.text(9.75, concat_y - 0.15, f'dim = {sum(dims)}', ha='center', va='center', fontsize=10, color='#ecf0f1')

ax.annotate('', xy=(12.0, concat_y), xytext=(11.1, concat_y),
            arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2))

box_nn = FancyBboxPatch((12.0, concat_y - 0.5), 3.0, 1.0, boxstyle='round,pad=0.15',
                        facecolor='#8e44ad', alpha=0.9, edgecolor='white', linewidth=2)
ax.add_patch(box_nn)
ax.text(13.5, concat_y + 0.15, 'Neural Network', ha='center', va='center', fontsize=12, fontweight='bold', color='white')
ax.text(13.5, concat_y - 0.15, '128→64→32→1(σ)', ha='center', va='center', fontsize=10, color='#ecf0f1')

box_target = FancyBboxPatch((12.0, concat_y - 2.3), 3.0, 1.0, boxstyle='round,pad=0.15',
                            facecolor='#27ae60', alpha=0.9, edgecolor='white', linewidth=2)
ax.add_patch(box_target)
ax.annotate('', xy=(13.5, concat_y - 1.3), xytext=(13.5, concat_y - 0.6),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))
ax.text(13.5, concat_y - 1.65, 'P(Plaintiff-', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
ax.text(13.5, concat_y - 1.95, 'Favorable)', ha='center', va='center', fontsize=11, fontweight='bold', color='white')

ax.text(0.5, 0.8, 'Binary Target: DISP 1 (Judgment for Plaintiff) + DISP 6 (Settled) + DISP 7 (Judgment on Consent)',
        fontsize=9, style='italic', color='#7f8c8d')
ax.text(0.5, 0.3, f'Total input dimensions: {sum(dims)} | Output: sigmoid ∈ [0, 1]',
        fontsize=9, style='italic', color='#7f8c8d')

plt.tight_layout()
plt.savefig('schema_pipeline.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Schema pipeline visualization saved.')

---
## Part 3: Seminal Case Library

Key precedents that shape the weighted prediction engine's doctrinal analysis.

In [ ]:
SEMINAL_CASES = {
    'concepcion': {
        'citation': 'AT&T Mobility LLC v. Concepcion, 563 U.S. 333 (2011)',
        'short_name': 'Concepcion',
        'year': 2011,
        'holding': 'FAA preempts state unconscionability rules that condition arbitration enforcement on class-action availability',
    },
    'cashcall': {
        'citation': 'CFPB v. CashCall, Inc., No. 1:13-cv-13167 (D. Mass. 2016)',
        'short_name': 'CashCall',
        'year': 2016,
        'holding': 'Lending practices violating state law constitute unfair/deceptive acts under UDAAP',
    },
    'overdraft_mdl': {
        'citation': 'In re Checking Account Overdraft Litigation, MDL No. 2036',
        'short_name': 'Overdraft MDL',
        'year': 2009,
        'holding': 'MDL consolidation of consumer banking claims creates aggregate settlement leverage',
    },
    'spokeo': {
        'citation': 'Spokeo, Inc. v. Robins, 578 U.S. 330 (2016)',
        'short_name': 'Spokeo',
        'year': 2016,
        'holding': 'Article III standing requires concrete injury, not merely statutory violation',
    },
    'transunion': {
        'citation': 'TransUnion LLC v. Ramirez, 594 U.S. 413 (2021)',
        'short_name': 'TransUnion',
        'year': 2021,
        'holding': 'Concrete injury must bear close relationship to traditionally recognized harms; each class member needs individual standing',
    },
    'marquette': {
        'citation': "Marquette Nat'l Bank v. First of Omaha Service Corp., 439 U.S. 299 (1978)",
        'short_name': 'Marquette',
        'year': 1978,
        'holding': 'National banks may export interest rates from home state regardless of borrower state usury limits',
    },
    'beneficial': {
        'citation': "Beneficial Nat'l Bank v. Anderson, 539 U.S. 1 (2003)",
        'short_name': 'Beneficial',
        'year': 2003,
        'holding': 'Complete preemption under National Bank Act allows removal of state-law usury claims to federal court',
    },
}

print('=== SEMINAL CASE LIBRARY ===')
for key, case in SEMINAL_CASES.items():
    print(f"  {case['short_name']} ({case['year']}): {case['holding'][:80]}...")

---
## Part 4: Load Real FJC Dataset

In [ ]:
CSV_PATH = 'attached_assets/fjc_sample_1771907678392.csv'

df_raw = pd.read_csv(CSV_PATH)
print(f'Raw dataset: {len(df_raw)} rows, {len(df_raw.columns)} columns')
print(f'Columns: {list(df_raw.columns)}')

df_raw['NOS_STR'] = df_raw['NOS'].astype(str).str.replace('.0', '', regex=False).str.strip()
df_raw['ORIGIN_STR'] = df_raw['ORIGIN'].astype(str).str.replace('.0', '', regex=False).str.strip()
df_raw['CIRCUIT_STR'] = df_raw['CIRCUIT'].astype(str).str.replace('.0', '', regex=False).str.strip()

df = df_raw[df_raw['NOS_STR'].isin(CONSUMER_BANKING_NOS_CODES)].copy()
print(f'\nFiltered to 4 consumer banking NOS codes: {len(df)} rows')
print(f'\nNOS distribution:')
for code in CONSUMER_BANKING_NOS_CODES:
    n = (df['NOS_STR'] == code).sum()
    print(f'  {code} ({NOS_CODE_DESCRIPTIONS[code]}): {n} ({n/len(df)*100:.1f}%)')

---
## Part 5: Feature Engineering & Binary Target

In [ ]:
df['YEAR'] = pd.to_datetime(df['FILEDATE']).dt.year

df['CLASSACT_BIN'] = (df['CLASSACT'] == 1.0).astype(int)

df['ARBIT_BIN'] = (df['ARBIT'].astype(str).str.strip() != '-8').astype(int)

df['MDL_BIN'] = (df['MDLDOCK'].astype(str).str.strip() != '-8').astype(int)

df['PLAINTIFF_FAVORABLE'] = df['DISP'].isin(PLAINTIFF_FAVORABLE_DISP).astype(int)

df['ERA'] = df['YEAR'].apply(get_doctrinal_era)

print('=== Binary Target Distribution ===')
pf_count = df['PLAINTIFF_FAVORABLE'].sum()
total = len(df)
print(f'Plaintiff-favorable (1): {pf_count} ({pf_count/total*100:.1f}%)')
print(f'Not plaintiff-favorable (0): {total - pf_count} ({(total-pf_count)/total*100:.1f}%)')

print(f'\nPlaintiff-favorable DISP breakdown:')
for disp in PLAINTIFF_FAVORABLE_DISP:
    n = (df['DISP'] == disp).sum()
    label = PLAINTIFF_FAVORABLE_LABELS.get(str(int(disp)), '')
    print(f'  DISP {int(disp)} ({label}): {n}')

print(f'\n=== Predictor Distributions ===')
print(f'CLASSACT=1 (class action): {df["CLASSACT_BIN"].sum()} ({df["CLASSACT_BIN"].mean()*100:.1f}%)')
print(f'ARBIT (has arbitration): {df["ARBIT_BIN"].sum()} ({df["ARBIT_BIN"].mean()*100:.1f}%)')
print(f'MDL (in MDL): {df["MDL_BIN"].sum()} ({df["MDL_BIN"].mean()*100:.1f}%)')
print(f'Year range: {df["YEAR"].min()} — {df["YEAR"].max()}')
print(f'\nDoctrinal era distribution:')
for i, name in enumerate(ERA_NAMES):
    n = (df['ERA'] == i).sum()
    if n > 0:
        pf = df[df['ERA'] == i]['PLAINTIFF_FAVORABLE'].mean()
        print(f'  {name}: {n} cases, {pf*100:.1f}% plaintiff-favorable')

### Dataset Visualizations

Distribution of key FJC variables and plaintiff-favorable outcomes across the dataset.

In [ ]:
fig = plt.figure(figsize=(20, 24))
gs = gridspec.GridSpec(4, 2, hspace=0.35, wspace=0.3)

# 1. NOS Code Distribution with PF rate
ax1 = fig.add_subplot(gs[0, 0])
nos_counts = []
nos_pf = []
nos_labels = []
for code in CONSUMER_BANKING_NOS_CODES:
    mask = df['NOS_STR'] == code
    nos_counts.append(mask.sum())
    nos_pf.append(df.loc[mask, 'PLAINTIFF_FAVORABLE'].mean() * 100 if mask.sum() > 0 else 0)
    nos_labels.append(f'{code}\n{NOS_CODE_DESCRIPTIONS[code].split("(")[0].strip()}')
bars = ax1.bar(nos_labels, nos_counts, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'], alpha=0.85, edgecolor='white')
ax1_twin = ax1.twinx()
ax1_twin.plot(nos_labels, nos_pf, 'ko-', markersize=8, linewidth=2, label='PF Rate %')
for i, (c, p) in enumerate(zip(nos_counts, nos_pf)):
    ax1.text(i, c + max(nos_counts)*0.02, f'{c}', ha='center', fontsize=9, fontweight='bold')
    ax1_twin.text(i, p + 1, f'{p:.1f}%', ha='center', fontsize=9, color='red', fontweight='bold')
ax1.set_title('NOS Code Distribution & Plaintiff-Favorable Rate', fontweight='bold')
ax1.set_ylabel('Case Count')
ax1_twin.set_ylabel('Plaintiff-Favorable Rate (%)', color='red')
ax1_twin.tick_params(axis='y', labelcolor='red')

# 2. Circuit Distribution
ax2 = fig.add_subplot(gs[0, 1])
circ_data = []
for circ in CIRCUIT_CODES:
    mask = df['CIRCUIT_STR'] == circ
    n = mask.sum()
    pf = df.loc[mask, 'PLAINTIFF_FAVORABLE'].mean() * 100 if n > 0 else 0
    label = CIRCUIT_CODE_LABELS.get(circ, circ)
    circ_data.append((label, n, pf))
circ_data.sort(key=lambda x: x[1], reverse=True)
c_labels, c_counts, c_pf = zip(*circ_data)
colors_circ = sns.color_palette('viridis', len(c_labels))
bars2 = ax2.barh(range(len(c_labels)), c_counts, color=colors_circ, alpha=0.85, edgecolor='white')
ax2.set_yticks(range(len(c_labels)))
ax2.set_yticklabels(c_labels, fontsize=9)
for i, (cnt, pf) in enumerate(zip(c_counts, c_pf)):
    ax2.text(cnt + max(c_counts)*0.02, i, f'{cnt} ({pf:.0f}% PF)', va='center', fontsize=8)
ax2.set_title('Cases by Federal Circuit', fontweight='bold')
ax2.set_xlabel('Case Count')
ax2.invert_yaxis()

# 3. Doctrinal Era Stacked Bar
ax3 = fig.add_subplot(gs[1, 0])
era_pf_counts = []
era_npf_counts = []
era_labels_plot = []
for i, name in enumerate(ERA_NAMES):
    mask = df['ERA'] == i
    n = mask.sum()
    if n > 0:
        pf_n = df.loc[mask, 'PLAINTIFF_FAVORABLE'].sum()
        era_pf_counts.append(pf_n)
        era_npf_counts.append(n - pf_n)
        era_labels_plot.append(name)
x_pos = range(len(era_labels_plot))
ax3.bar(x_pos, era_npf_counts, label='Not PF', color='#e74c3c', alpha=0.7, edgecolor='white')
ax3.bar(x_pos, era_pf_counts, bottom=era_npf_counts, label='Plaintiff-Favorable', color='#27ae60', alpha=0.85, edgecolor='white')
for i, (npf, pf) in enumerate(zip(era_npf_counts, era_pf_counts)):
    total = npf + pf
    ax3.text(i, total + max(era_npf_counts)*0.03, f'{pf/(npf+pf)*100:.1f}%', ha='center', fontsize=10, fontweight='bold', color='#27ae60')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(era_labels_plot, fontsize=9)
ax3.set_title('Outcomes by Doctrinal Era', fontweight='bold')
ax3.set_ylabel('Case Count')
ax3.legend(loc='upper right')

# 4. DISP Code Distribution
ax4 = fig.add_subplot(gs[1, 1])
disp_vals = sorted(df['DISP'].dropna().unique())
disp_counts_plot = []
disp_labels_plot = []
disp_colors = []
for d in disp_vals:
    n = (df['DISP'] == d).sum()
    if n > 5:
        label = DISPOSITION_CODE_LABELS.get(str(int(d)), str(int(d)))
        disp_counts_plot.append(n)
        disp_labels_plot.append(f'DISP {int(d)}')
        disp_colors.append('#27ae60' if d in PLAINTIFF_FAVORABLE_DISP else '#95a5a6')
ax4.barh(range(len(disp_labels_plot)), disp_counts_plot, color=disp_colors, alpha=0.85, edgecolor='white')
ax4.set_yticks(range(len(disp_labels_plot)))
ax4.set_yticklabels(disp_labels_plot, fontsize=9)
for i, cnt in enumerate(disp_counts_plot):
    ax4.text(cnt + max(disp_counts_plot)*0.02, i, f'{cnt}', va='center', fontsize=8)
ax4.set_title('Disposition Code Distribution', fontweight='bold')
ax4.set_xlabel('Count')
green_patch = mpatches.Patch(color='#27ae60', alpha=0.85, label='Plaintiff-Favorable')
gray_patch = mpatches.Patch(color='#95a5a6', alpha=0.85, label='Not Plaintiff-Favorable')
ax4.legend(handles=[green_patch, gray_patch], loc='lower right', fontsize=9)
ax4.invert_yaxis()

# 5. Binary Target Pie
ax5 = fig.add_subplot(gs[2, 0])
pf_total = int(df['PLAINTIFF_FAVORABLE'].sum())
npf_total = len(df) - pf_total
wedges, texts, autotexts = ax5.pie(
    [npf_total, pf_total],
    labels=['Not Plaintiff-Favorable', 'Plaintiff-Favorable'],
    colors=['#e74c3c', '#27ae60'],
    autopct=lambda p: f'{p:.1f}%\n({int(p*len(df)/100)})',
    startangle=90,
    explode=(0, 0.08),
    textprops={'fontsize': 11}
)
ax5.set_title('Binary Target Distribution', fontweight='bold')

# 6. Year Filed Histogram with PF overlay
ax6 = fig.add_subplot(gs[2, 1])
year_bins = range(int(df['YEAR'].min()), int(df['YEAR'].max()) + 2)
ax6.hist(df[df['PLAINTIFF_FAVORABLE'] == 0]['YEAR'], bins=year_bins, alpha=0.6, color='#e74c3c', label='Not PF', edgecolor='white')
ax6.hist(df[df['PLAINTIFF_FAVORABLE'] == 1]['YEAR'], bins=year_bins, alpha=0.8, color='#27ae60', label='Plaintiff-Favorable', edgecolor='white')
for yr, label in [(2011, 'Concepcion'), (2016, 'Spokeo'), (2021, 'TransUnion')]:
    ax6.axvline(x=yr, color='#2c3e50', linestyle='--', alpha=0.7, linewidth=1.5)
    ax6.text(yr, ax6.get_ylim()[1]*0.95, f' {label}', fontsize=8, rotation=90, va='top', color='#2c3e50', fontweight='bold')
ax6.set_title('Filing Year Distribution with Doctrinal Milestones', fontweight='bold')
ax6.set_xlabel('Year Filed')
ax6.set_ylabel('Case Count')
ax6.legend()

# 7. Class Action / Arbitration / MDL rates by era
ax7 = fig.add_subplot(gs[3, 0])
era_ca = []
era_arb = []
era_mdl = []
era_x = []
for i, name in enumerate(ERA_NAMES):
    mask = df['ERA'] == i
    if mask.sum() > 0:
        era_x.append(name)
        era_ca.append(df.loc[mask, 'CLASSACT_BIN'].mean() * 100)
        era_arb.append(df.loc[mask, 'ARBIT_BIN'].mean() * 100)
        era_mdl.append(df.loc[mask, 'MDL_BIN'].mean() * 100)
x_idx = np.arange(len(era_x))
w = 0.25
ax7.bar(x_idx - w, era_ca, w, label='Class Action', color='#3498db', alpha=0.85)
ax7.bar(x_idx, era_arb, w, label='Arbitration', color='#e74c3c', alpha=0.85)
ax7.bar(x_idx + w, era_mdl, w, label='MDL', color='#f39c12', alpha=0.85)
ax7.set_xticks(x_idx)
ax7.set_xticklabels(era_x, fontsize=9)
ax7.set_title('Procedural Feature Rates by Doctrinal Era', fontweight='bold')
ax7.set_ylabel('Rate (%)')
ax7.legend()

# 8. Origin Distribution
ax8 = fig.add_subplot(gs[3, 1])
orig_data = []
for orig in ORIGIN_CODES:
    mask = df['ORIGIN_STR'] == orig
    n = mask.sum()
    if n > 0:
        pf = df.loc[mask, 'PLAINTIFF_FAVORABLE'].mean() * 100
        label = ORIGIN_CODE_LABELS.get(orig, orig)[:25]
        orig_data.append((f'{orig}: {label}', n, pf))
orig_data.sort(key=lambda x: x[1], reverse=True)
o_labels, o_counts, o_pf = zip(*orig_data)
bars8 = ax8.barh(range(len(o_labels)), o_counts, color=sns.color_palette('Set2', len(o_labels)), alpha=0.85, edgecolor='white')
ax8.set_yticks(range(len(o_labels)))
ax8.set_yticklabels(o_labels, fontsize=9)
for i, (cnt, pf) in enumerate(zip(o_counts, o_pf)):
    ax8.text(cnt + max(o_counts)*0.02, i, f'{cnt} ({pf:.0f}% PF)', va='center', fontsize=8)
ax8.set_title('Case Origin Distribution', fontweight='bold')
ax8.set_xlabel('Count')
ax8.invert_yaxis()

plt.savefig('dataset_visualizations.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Dataset visualizations saved.')

---
## Part 6: Feature Encoding for Neural Network

In [ ]:
def encode_case(nos_str, origin_str, class_action, circuit_str, year, has_arbit, has_mdl):
    """Encode a single case into a feature vector for the neural network.
    
    Args:
        nos_str: NOS code string ('480', '190', '371', '370')
        origin_str: Origin code string ('1'-'8')
        class_action: Boolean, CLASSACT field
        circuit_str: Circuit code string ('0'-'11')
        year: Filing year integer
        has_arbit: Boolean, ARBIT field
        has_mdl: Boolean, MDLDOCK field
    
    Returns:
        numpy array of shape (INPUT_DIM,) = (32,)
    """
    features = np.zeros(INPUT_DIM, dtype=np.float32)
    offset = 0
    
    if nos_str in CONSUMER_BANKING_NOS_CODES:
        features[offset + CONSUMER_BANKING_NOS_CODES.index(nos_str)] = 1.0
    offset += N_NOS
    
    if origin_str in ORIGIN_CODES:
        features[offset + ORIGIN_CODES.index(origin_str)] = 1.0
    offset += N_ORIGIN
    
    if circuit_str in CIRCUIT_CODES:
        features[offset + CIRCUIT_CODES.index(circuit_str)] = 1.0
    offset += N_CIRCUIT
    
    features[offset] = 1.0 if class_action else 0.0
    offset += N_CLASS_ACTION
    
    features[offset] = 1.0 if has_arbit else 0.0
    offset += N_ARBIT
    
    features[offset] = 1.0 if has_mdl else 0.0
    offset += N_MDL
    
    features[offset] = (year - 1985) / 40.0
    offset += N_YEAR
    
    era = get_doctrinal_era(year)
    features[offset + era] = 1.0
    
    return features

test_vec = encode_case('480', '1', False, '2', 2020, False, False)
print(f'Test encoding shape: {test_vec.shape} (expect {INPUT_DIM})')
print(f'Test encoding sum: {test_vec.sum():.2f}')
print(f'Non-zero features: {np.count_nonzero(test_vec)}')

In [ ]:
print('Encoding full dataset...')

X = np.array([
    encode_case(
        row['NOS_STR'],
        row['ORIGIN_STR'],
        bool(row['CLASSACT_BIN']),
        row['CIRCUIT_STR'],
        int(row['YEAR']),
        bool(row['ARBIT_BIN']),
        bool(row['MDL_BIN']),
    )
    for _, row in df.iterrows()
], dtype=np.float32)

y = df['PLAINTIFF_FAVORABLE'].values.astype(np.float32)

print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')
print(f'Target distribution: {int(y.sum())} positive / {int(len(y) - y.sum())} negative')
print(f'Positive rate: {y.mean()*100:.1f}%')

np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(0.8 * len(X))

X_train = X[indices[:split]]
y_train = y[indices[:split]]
X_val = X[indices[split:]]
y_val = y[indices[split:]]

print(f'\nTrain: {len(X_train)} samples ({y_train.mean()*100:.1f}% positive)')
print(f'Val:   {len(X_val)} samples ({y_val.mean()*100:.1f}% positive)')

---
## Part 7: Neural Network Architecture

Architecture: **32 → 128 → 64 → 32 → 1 (sigmoid)**

- Binary classification (plaintiff-favorable vs. not)
- Batch normalization after each dense layer
- Dropout for regularization
- Class weights to handle ~9% positive rate imbalance

In [ ]:
def build_model(input_dim, learning_rate=0.001):
    """Build the FJC litigation prediction neural network.
    
    Architecture: input_dim → 128 → 64 → 32 → 1 (sigmoid)
    Each hidden layer: Dense → BatchNorm → ReLU → Dropout
    """
    inputs = keras.Input(shape=(input_dim,), name='case_features')
    
    x = layers.Dense(128, name='dense_1')(inputs)
    x = layers.BatchNormalization(name='bn_1')(x)
    x = layers.Activation('relu', name='relu_1')(x)
    x = layers.Dropout(0.3, name='dropout_1')(x)
    
    x = layers.Dense(64, name='dense_2')(x)
    x = layers.BatchNormalization(name='bn_2')(x)
    x = layers.Activation('relu', name='relu_2')(x)
    x = layers.Dropout(0.25, name='dropout_2')(x)
    
    x = layers.Dense(32, name='dense_3')(x)
    x = layers.BatchNormalization(name='bn_3')(x)
    x = layers.Activation('relu', name='relu_3')(x)
    x = layers.Dropout(0.2, name='dropout_3')(x)
    
    outputs = layers.Dense(1, activation='sigmoid', name='plaintiff_favorable')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='fjc_litigation_predictor')
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ],
    )
    
    return model

model = build_model(INPUT_DIM)
model.summary()

### Neural Network Architecture Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(-1, 17)
ax.set_ylim(-1, 9)
ax.axis('off')
ax.set_title('Neural Network Architecture: FJC Litigation Binary Classifier', fontsize=16, fontweight='bold', pad=20)

layer_info = [
    ('Input\n32 dims', 32, '#3498db', 0),
    ('Dense\n128', 128, '#e74c3c', 3),
    ('Dense\n64', 64, '#f39c12', 6),
    ('Dense\n32', 32, '#9b59b6', 9),
    ('Output\nσ(1)', 1, '#27ae60', 12),
]

max_neurons = 128
for name, neurons, color, x in layer_info:
    h = max(0.8, (neurons / max_neurons) * 6)
    y_center = 4
    rect = FancyBboxPatch((x, y_center - h/2), 2, h, boxstyle='round,pad=0.15',
                          facecolor=color, alpha=0.85, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 1, y_center, name, ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')

annotations = [
    (1, 'BatchNorm\nReLU\nDropout 0.3', '#e74c3c'),
    (2, 'BatchNorm\nReLU\nDropout 0.25', '#f39c12'),
    (3, 'BatchNorm\nReLU\nDropout 0.2', '#9b59b6'),
]

for idx, annot_text, color in annotations:
    x_pos = layer_info[idx][3] + 1
    ax.text(x_pos, 0.8, annot_text, ha='center', va='center',
            fontsize=8, color=color, style='italic',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.1))

for i in range(len(layer_info) - 1):
    x1 = layer_info[i][3] + 2
    x2 = layer_info[i+1][3]
    ax.annotate('', xy=(x2, 4), xytext=(x1, 4),
                arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=2.5))
    mid_x = (x1 + x2) / 2
    ax.text(mid_x, 4.5, f'{layer_info[i][1]}→{layer_info[i+1][1]}',
            ha='center', fontsize=8, color='#7f8c8d')

ax.text(13, 1.5, 'Loss: Binary Crossentropy\nOptimizer: Adam (lr=0.001)\nClass Weights: Balanced',
        fontsize=9, ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#ecf0f1', edgecolor='#bdc3c7'))

input_labels = ['NOS (4)', 'ORIGIN (8)', 'CIRCUIT (12)', 'CLASSACT (1)',
                'ARBIT (1)', 'MDL (1)', 'YEAR (1)', 'ERA (4)']
for i, label in enumerate(input_labels):
    y = 7.8 - i * 0.7
    ax.text(-0.5, y, label, fontsize=7, ha='right', va='center', color='#3498db', fontweight='bold')
    ax.annotate('', xy=(0, y), xytext=(-0.3, y),
                arrowprops=dict(arrowstyle='->', color='#3498db', lw=0.8))

ax.text(15, 4, 'P(Plaintiff\nFavorable)',
        fontsize=11, ha='center', va='center', fontweight='bold', color='#27ae60',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#27ae60', alpha=0.15))
ax.annotate('', xy=(14.2, 4), xytext=(14, 4),
            arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2))

plt.tight_layout()
plt.savefig('nn_architecture.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('NN architecture diagram saved.')

---
## Part 8: Training with Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train
)
class_weight_dict = {0: class_weights_arr[0], 1: class_weights_arr[1]}
print(f'Class weights: {class_weight_dict}')
print(f'  Weight for negative (0): {class_weights_arr[0]:.3f}')
print(f'  Weight for positive (1): {class_weights_arr[1]:.3f}')

cb = [
    callbacks.EarlyStopping(
        monitor='val_auc',
        patience=15,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=cb,
    verbose=1
)

### Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Loss
ax = axes[0, 0]
ax.plot(history.history['loss'], label='Train Loss', linewidth=2, color='#3498db')
ax.plot(history.history['val_loss'], label='Val Loss', linewidth=2, color='#e74c3c')
ax.set_title('Loss (Binary Crossentropy)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# AUC
ax = axes[0, 1]
ax.plot(history.history['auc'], label='Train AUC', linewidth=2, color='#3498db')
ax.plot(history.history['val_auc'], label='Val AUC', linewidth=2, color='#e74c3c')
ax.set_title('AUC (Area Under ROC Curve)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy
ax = axes[1, 0]
ax.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2, color='#3498db')
ax.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2, color='#e74c3c')
ax.set_title('Accuracy', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# Precision & Recall
ax = axes[1, 1]
ax.plot(history.history['precision'], label='Train Precision', linewidth=2, color='#3498db', linestyle='-')
ax.plot(history.history['val_precision'], label='Val Precision', linewidth=2, color='#e74c3c', linestyle='-')
ax.plot(history.history['recall'], label='Train Recall', linewidth=2, color='#3498db', linestyle='--')
ax.plot(history.history['val_recall'], label='Val Recall', linewidth=2, color='#e74c3c', linestyle='--')
ax.set_title('Precision & Recall', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Score')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Neural Network Training History', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Training curves saved.')

---
## Part 9: Neural Network Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

results = model.evaluate(X_val, y_val, verbose=0, return_dict=True)
print('=== Validation Metrics ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')

y_pred_prob = model.predict(X_val, verbose=0).flatten()
y_pred = (y_pred_prob >= 0.5).astype(int)

print(f'\n=== ROC AUC Score ===')
print(f'  {roc_auc_score(y_val, y_pred_prob):.4f}')

print(f'\n=== Confusion Matrix ===')
cm = confusion_matrix(y_val, y_pred)
print(f'  TN={cm[0,0]:5d}  FP={cm[0,1]:5d}')
print(f'  FN={cm[1,0]:5d}  TP={cm[1,1]:5d}')

print(f'\n=== Classification Report ===')
print(classification_report(y_val, y_pred, target_names=['Not Plaintiff-Favorable', 'Plaintiff-Favorable'], zero_division=0))

### Evaluation Visualizations: Confusion Matrix & ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix

if 'y_pred_prob' not in dir():
    y_pred_prob = model.predict(X_val, verbose=0).flatten()
    y_pred = (y_pred_prob >= 0.5).astype(int)


fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Confusion Matrix Heatmap
ax = axes[0]
cm = confusion_matrix(y_val, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not PF', 'Plaintiff-Fav'],
            yticklabels=['Not PF', 'Plaintiff-Fav'],
            cbar_kws={'label': 'Count'},
            linewidths=2, linecolor='white')
for i in range(2):
    for j in range(2):
        ax.text(j + 0.5, i + 0.75, f'({cm_pct[i, j]:.1f}%)',
                ha='center', va='center', fontsize=9, color='gray')
ax.set_title('Confusion Matrix', fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')

# 2. ROC Curve
ax = axes[1]
fpr, tpr, thresholds = roc_curve(y_val, y_pred_prob)
roc_auc = auc(fpr, tpr)
ax.plot(fpr, tpr, color='#3498db', linewidth=2.5, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.fill_between(fpr, tpr, alpha=0.15, color='#3498db')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, linewidth=1, label='Random (AUC = 0.5)')
ax.set_title('ROC Curve', fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# 3. Prediction Distribution
ax = axes[2]
ax.hist(y_pred_prob[y_val == 0], bins=30, alpha=0.7, color='#e74c3c', label='Actual: Not PF', edgecolor='white')
ax.hist(y_pred_prob[y_val == 1], bins=30, alpha=0.7, color='#27ae60', label='Actual: Plaintiff-Fav', edgecolor='white')
ax.axvline(x=0.5, color='#2c3e50', linestyle='--', linewidth=2, label='Threshold (0.5)')
ax.set_title('Predicted Probability Distribution', fontweight='bold')
ax.set_xlabel('Predicted P(Plaintiff-Favorable)')
ax.set_ylabel('Count')
ax.legend()

plt.suptitle('Model Evaluation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evaluation_visualizations.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Evaluation visualizations saved.')

In [ ]:
print('=== Plaintiff-Favorable Rate by Feature (Validation Set) ===')

val_df = df.iloc[indices[split:]].copy()
val_df['PRED_PROB'] = y_pred_prob
val_df['PRED'] = y_pred

print(f'\nBy NOS Code:')
for code in CONSUMER_BANKING_NOS_CODES:
    mask = val_df['NOS_STR'] == code
    if mask.sum() > 0:
        actual = val_df.loc[mask, 'PLAINTIFF_FAVORABLE'].mean()
        predicted = val_df.loc[mask, 'PRED_PROB'].mean()
        print(f'  {code} ({NOS_CODE_DESCRIPTIONS[code]}): actual={actual*100:.1f}% predicted={predicted*100:.1f}% (n={mask.sum()})')

print(f'\nBy Arbitration:')
for arb in [0, 1]:
    mask = val_df['ARBIT_BIN'] == arb
    if mask.sum() > 0:
        actual = val_df.loc[mask, 'PLAINTIFF_FAVORABLE'].mean()
        predicted = val_df.loc[mask, 'PRED_PROB'].mean()
        label = 'No arbitration' if arb == 0 else 'Has arbitration'
        print(f'  {label}: actual={actual*100:.1f}% predicted={predicted*100:.1f}% (n={mask.sum()})')

print(f'\nBy Class Action:')
for ca in [0, 1]:
    mask = val_df['CLASSACT_BIN'] == ca
    if mask.sum() > 0:
        actual = val_df.loc[mask, 'PLAINTIFF_FAVORABLE'].mean()
        predicted = val_df.loc[mask, 'PRED_PROB'].mean()
        label = 'No class action' if ca == 0 else 'Class action'
        print(f'  {label}: actual={actual*100:.1f}% predicted={predicted*100:.1f}% (n={mask.sum()})')

print(f'\nBy Doctrinal Era:')
for i, name in enumerate(ERA_NAMES):
    mask = val_df['ERA'] == i
    if mask.sum() > 0:
        actual = val_df.loc[mask, 'PLAINTIFF_FAVORABLE'].mean()
        predicted = val_df.loc[mask, 'PRED_PROB'].mean()
        print(f'  {name}: actual={actual*100:.1f}% predicted={predicted*100:.1f}% (n={mask.sum()})')

print(f'\nBy Circuit:')
for circ in sorted(val_df['CIRCUIT_STR'].unique()):
    mask = val_df['CIRCUIT_STR'] == circ
    if mask.sum() > 5:
        actual = val_df.loc[mask, 'PLAINTIFF_FAVORABLE'].mean()
        predicted = val_df.loc[mask, 'PRED_PROB'].mean()
        label = CIRCUIT_CODE_LABELS.get(circ, f'Circuit {circ}')
        print(f'  {label}: actual={actual*100:.1f}% predicted={predicted*100:.1f}% (n={mask.sum()})')

---
## Part 10: Weighted Prediction Engine (Python Port)

This is the complete Python translation of `server/prediction-engine.ts` from the web app.
It provides the weighted, doctrinal-analysis-based prediction model with:
- Per-factor impact scoring with seminal case explanations
- Interaction effects (e.g., Concepcion + class action, TransUnion + no concrete injury)
- Outcome distribution (dismissal, settlement, plaintiff/defendant judgment)
- Settlement range estimation
- Risk assessment
- Strategic notes generation

In [ ]:
CIRCUIT_PLAINTIFF_FAVORABILITY = {
    '1': 0.52, '2': 0.55, '3': 0.50, '4': 0.45, '5': 0.38,
    '6': 0.48, '7': 0.50, '8': 0.40, '9': 0.58, '10': 0.44,
    '11': 0.46, 'DC': 0.54, '0': 0.54,
}

NOS_BASE_WIN_RATES = {'480': 0.42, '190': 0.40, '371': 0.45, '370': 0.38}

STANDING_SENSITIVE_NOS = ['480', '371', '370']
MDL_FAVORABLE_NOS = ['370', '371', '190', '480']

MAJOR_BANK_RISK = {
    'JPMorgan Chase': 0.04, 'Bank of America': 0.05, 'Wells Fargo': 0.06,
    'Citibank': 0.04, 'Capital One': 0.03, 'U.S. Bank': 0.02,
    'PNC Financial': 0.02, 'TD Bank': 0.03, 'Truist': 0.01,
    'Goldman Sachs': -0.02, 'Morgan Stanley': -0.02, 'Ally Financial': 0.03,
    'Discover': 0.02, 'Synchrony Financial': 0.04, 'CashCall / Western Sky': 0.10,
}


def clamp(val, lo, hi):
    return max(lo, min(hi, val))

def rnd(val):
    return round(val * 1000) / 1000


def predict_outcome_weighted(case_name, nos_code, origin_code, class_action, circuit_code,
                             year_filed, has_arbitration=False, concrete_injury=True,
                             is_mdl=False, prior_regulatory=False, defendant_bank='Other'):
    """Full weighted prediction engine — Python port of server/prediction-engine.ts.
    
    Returns a dict matching the PredictionResult interface.
    """
    factors = []
    era = get_doctrinal_era_name(year_filed)
    
    score = NOS_BASE_WIN_RATES.get(nos_code, 0.35)
    
    nos_label = NOS_CODE_DESCRIPTIONS.get(nos_code, NOS_CODE_LABELS.get(nos_code, f'NOS {nos_code}'))
    nos_base = NOS_BASE_WIN_RATES.get(nos_code, 0.35)
    nos_dev = nos_base - 0.38
    nos_explanations = {
        '480': f"Consumer Credit claims (FCRA, FDCPA, EFTA) carry a moderate-to-favorable base rate. Statutory claims provide fee-shifting provisions, but post-{SEMINAL_CASES['spokeo']['short_name']}/{SEMINAL_CASES['transunion']['short_name']} standing requirements create new barriers.",
        '371': f"Truth in Lending claims have strict liability provisions under TILA/Reg Z. Disclosure violations carry clear remedial schemes.",
        '190': f"Banks and Banking contract claims involve complex regulatory frameworks where preemption under {SEMINAL_CASES['beneficial']['short_name']} and rate exportation under {SEMINAL_CASES['marquette']['short_name']} can be dispositive.",
        '370': f"Other Fraud/UDAAP claims follow the {SEMINAL_CASES['cashcall']['short_name']} framework. CFPB enforcement established that lending practices violating state law constitute unfair/deceptive acts.",
    }
    factors.append({
        'factor': f'NOS Code: {nos_code} — {nos_label}',
        'impact': rnd(nos_dev),
        'direction': 'favorable' if nos_dev >= 0 else 'unfavorable',
        'explanation': nos_explanations.get(nos_code, 'Historical FJC data informs baseline.'),
    })
    score += nos_dev
    
    circ_fav = CIRCUIT_PLAINTIFF_FAVORABILITY.get(circuit_code, 0.48)
    circ_impact = (circ_fav - 0.48) * 0.4
    circ_label = CIRCUIT_CODE_LABELS.get(circuit_code, f'Circuit {circuit_code}')
    factors.append({
        'factor': f'Circuit: {circ_label}',
        'impact': rnd(circ_impact),
        'direction': 'favorable' if circ_impact >= 0.01 else ('unfavorable' if circ_impact <= -0.01 else 'neutral'),
        'explanation': f'{circ_label} plaintiff-favorability index: {int(circ_fav*100)}%.',
    })
    score += circ_impact
    
    origin_label = ORIGIN_CODE_LABELS.get(origin_code, f'Origin {origin_code}')
    origin_impact = 0
    origin_exp = f'Case origin: {origin_label}. '
    if is_removed_from_state_court(origin_code):
        origin_impact = -0.06
        origin_exp += f"Removal typically disadvantages consumer plaintiffs. Under {SEMINAL_CASES['beneficial']['short_name']}, national banks can remove under complete preemption."
    elif is_mdl_origin(origin_code):
        origin_impact = 0.04
        origin_exp += f"MDL origin per {SEMINAL_CASES['overdraft_mdl']['short_name']} creates economies of scale."
    elif origin_code == '1':
        origin_impact = 0.02
        origin_exp += 'Original federal filing suggests plaintiff chose federal forum for strongest statutory claims.'
    factors.append({
        'factor': f'Origin: {origin_label}',
        'impact': rnd(origin_impact),
        'direction': 'favorable' if origin_impact > 0 else ('unfavorable' if origin_impact < 0 else 'neutral'),
        'explanation': origin_exp,
    })
    score += origin_impact
    
    if class_action:
        if has_arbitration:
            ca_impact = -0.06
            ca_exp = f"Class action + arbitration clause. Under {SEMINAL_CASES['concepcion']['citation']}, class action waivers are enforceable, eliminating the primary vehicle for small-dollar claims."
            ca_label = 'Class Action + Arbitration Conflict (Concepcion)'
        else:
            ca_impact = 0.08
            ca_exp = 'Class action increases settlement pressure and plaintiff leverage through aggregate claims.'
            ca_label = 'Class Action Allegation'
        factors.append({
            'factor': ca_label,
            'impact': rnd(ca_impact),
            'direction': 'favorable' if ca_impact > 0 else 'unfavorable',
            'explanation': ca_exp,
        })
        score += ca_impact
    
    era_impacts = {
        'pre-concepcion': (0.08, f"Before {SEMINAL_CASES['concepcion']['short_name']}. Most favorable era for consumer class actions."),
        'post-concepcion-pre-spokeo': (0.02, f"After Concepcion but before Spokeo. Arbitration tightened, but standing not yet constricted."),
        'post-spokeo-pre-transunion': (-0.04, f"After {SEMINAL_CASES['spokeo']['short_name']}. Concrete-injury requirement created uncertainty for statutory claims."),
        'post-transunion': (-0.08, f"After {SEMINAL_CASES['transunion']['short_name']}. Tightest standing + robust arbitration = most challenging landscape."),
    }
    era_imp, era_exp = era_impacts.get(era, (0, ''))
    factors.append({
        'factor': f'Doctrinal Era: {era.replace("-", " ")}',
        'impact': rnd(era_imp),
        'direction': 'favorable' if era_imp >= 0 else 'unfavorable',
        'explanation': era_exp,
    })
    score += era_imp
    
    if not has_arbitration:
        arb_impact = 0.06
        arb_exp = f"No arbitration clause — retains access to court, discovery, class certification, and jury trial."
        arb_label = 'No Arbitration Clause'
    else:
        arb_impact = -0.08 if era == 'pre-concepcion' else -0.18
        arb_exp = f"Under {SEMINAL_CASES['concepcion']['citation']}, mandatory arbitration with class-action waiver is enforceable. Dominant variable in consumer banking litigation."
        arb_label = 'Mandatory Arbitration Clause (Concepcion)'
    factors.append({
        'factor': arb_label,
        'impact': rnd(arb_impact),
        'direction': 'favorable' if arb_impact > 0 else 'unfavorable',
        'explanation': arb_exp,
    })
    score += arb_impact
    
    is_sensitive = nos_code in STANDING_SENSITIVE_NOS
    if is_sensitive and era in ('post-spokeo-pre-transunion', 'post-transunion'):
        if concrete_injury:
            standing_impact = 0.04
            factors.append({
                'factor': 'Concrete Injury Documented (Standing Cleared)',
                'impact': standing_impact,
                'direction': 'favorable',
                'explanation': f"Documented concrete injury satisfies {SEMINAL_CASES['spokeo']['short_name']}/{SEMINAL_CASES['transunion']['short_name']} standing requirement.",
            })
        else:
            standing_impact = -0.14 if era == 'post-transunion' else -0.08
            factors.append({
                'factor': f'Standing Risk: No Concrete Injury ({"TransUnion" if era == "post-transunion" else "Spokeo"})',
                'impact': rnd(standing_impact),
                'direction': 'unfavorable',
                'explanation': f"Standing-sensitive claim without concrete injury faces Article III barriers under {SEMINAL_CASES['transunion']['citation']}.",
            })
        score += standing_impact
    
    if is_mdl or is_mdl_origin(origin_code):
        is_fav_nos = nos_code in MDL_FAVORABLE_NOS
        mdl_impact = 0.08 if is_fav_nos else 0.04
        factors.append({
            'factor': 'MDL Consolidation',
            'impact': rnd(mdl_impact),
            'direction': 'favorable',
            'explanation': f"MDL consolidation per {SEMINAL_CASES['overdraft_mdl']['citation']} creates plaintiff advantages: coordinated discovery, bellwether trials, aggregate settlement pressure.",
        })
        score += mdl_impact
    
    if prior_regulatory:
        reg_impact = 0.08
        factors.append({
            'factor': 'Prior/Parallel Regulatory Action',
            'impact': reg_impact,
            'direction': 'favorable',
            'explanation': f"Regulatory enforcement (CFPB, OCC, state AG) per {SEMINAL_CASES['cashcall']['short_name']} model establishes liability standards for private claims.",
        })
        score += reg_impact
    
    bank_risk = MAJOR_BANK_RISK.get(defendant_bank)
    if bank_risk is not None:
        factors.append({
            'factor': f'Defendant: {defendant_bank}',
            'impact': rnd(bank_risk),
            'direction': 'favorable' if bank_risk > 0 else ('unfavorable' if bank_risk < 0 else 'neutral'),
            'explanation': f'{defendant_bank} risk profile based on FJC disposition history and regulatory enforcement frequency.',
        })
        score += bank_risk
    
    if class_action and is_mdl and not has_arbitration:
        factors.append({
            'factor': 'MDL + Class Action + No Arbitration',
            'impact': 0.06,
            'direction': 'favorable',
            'explanation': f"Strongest procedural posture: MDL consolidation, class mechanism intact, no arbitration barrier.",
        })
        score += 0.06
    
    if nos_code == '370' and prior_regulatory and defendant_bank == 'CashCall / Western Sky':
        factors.append({
            'factor': 'UDAAP + Regulatory Action + CashCall Alignment',
            'impact': 0.08,
            'direction': 'favorable',
            'explanation': f"Direct alignment with {SEMINAL_CASES['cashcall']['citation']}.",
        })
        score += 0.08
    
    if is_sensitive and not concrete_injury and class_action and era == 'post-transunion':
        factors.append({
            'factor': 'TransUnion + Class Action + No Concrete Injury',
            'impact': -0.10,
            'direction': 'unfavorable',
            'explanation': f"Most challenging interaction: {SEMINAL_CASES['transunion']['citation']} requires individual standing for each class member.",
        })
        score -= 0.10
    
    win_prob = clamp(score, 0.05, 0.92)
    
    dismissal = 0.22
    if has_arbitration: dismissal += 0.20
    if is_removed_from_state_court(origin_code): dismissal += 0.06
    if is_sensitive and not concrete_injury and era == 'post-transunion': dismissal += 0.12
    if class_action and not has_arbitration: dismissal -= 0.06
    if prior_regulatory: dismissal -= 0.05
    dismissal -= (win_prob - 0.5) * 0.15
    dismissal = clamp(dismissal, 0.05, 0.75)
    
    settlement = 0.38
    if is_mdl or is_mdl_origin(origin_code): settlement += 0.10
    if class_action and not has_arbitration: settlement += 0.08
    if prior_regulatory: settlement += 0.08
    if win_prob > 0.55: settlement += 0.06
    if has_arbitration: settlement -= 0.18
    settlement = clamp(settlement, 0.10, 0.70)
    
    remaining = clamp(1 - dismissal - settlement, 0.05, 0.50)
    plaintiff_judgment = remaining * win_prob
    defendant_judgment = remaining * (1 - win_prob)
    
    base_damages = 5_000_000 if class_action else 250_000
    if is_mdl: base_damages *= 3
    multiplier = win_prob * 0.5
    settlement_low = max(int(base_damages * multiplier * 0.2), 5000)
    settlement_high = max(int(base_damages * multiplier * 0.7), settlement_low + 10000)
    
    if win_prob >= 0.70: risk = 'Very High'
    elif win_prob >= 0.55: risk = 'High'
    elif win_prob >= 0.40: risk = 'Moderate'
    elif win_prob >= 0.25: risk = 'Low'
    else: risk = 'Very Low'
    
    notes = []
    if has_arbitration:
        notes.append(f"CRITICAL: Under {SEMINAL_CASES['concepcion']['citation']}, the mandatory arbitration clause is the dominant variable. Evaluate unconscionability challenges, delegation clause severability, effective vindication doctrine, and McGill exceptions.")
    if is_sensitive and not concrete_injury and era in ('post-spokeo-pre-transunion', 'post-transunion'):
        notes.append(f"CRITICAL — STANDING BARRIER: Under {SEMINAL_CASES['transunion']['citation']}, this claim type faces substantial Article III standing barriers without documented concrete injury.")
    if is_removed_from_state_court(origin_code):
        notes.append(f"Case removed from state court — defense likely seeks federal forum advantages. Consider remand motion if state-law claims predominate.")
    if is_mdl or is_mdl_origin(origin_code):
        notes.append(f"MDL consolidation follows the {SEMINAL_CASES['overdraft_mdl']['short_name']} paradigm. Coordinate with lead counsel and leverage shared discovery efficiencies.")
    if prior_regulatory:
        notes.append(f"Prior regulatory action provides leverage. Monitor proceedings for collateral estoppel opportunities.")
    if class_action and not has_arbitration and win_prob > 0.5:
        notes.append(f"Class action without arbitration barrier — focus on certification requirements: numerosity, commonality, typicality, adequacy.")
    if era == 'post-transunion' and win_prob < 0.40:
        notes.append(f"Consider alternative dispute resolution or state court filing to avoid federal standing requirements.")
    
    return {
        'id': str(uuid.uuid4()),
        'caseName': case_name,
        'timestamp': datetime.now().isoformat(),
        'plaintiffFavorableProbability': rnd(win_prob),
        'dismissalProbability': rnd(dismissal),
        'settlementProbability': rnd(settlement),
        'plaintiffJudgmentProbability': rnd(plaintiff_judgment),
        'defendantJudgmentProbability': rnd(defendant_judgment),
        'expectedSettlementRange': {'low': settlement_low, 'high': settlement_high},
        'keyFactors': factors,
        'riskAssessment': risk,
        'strategicNotes': notes,
    }

print('Weighted prediction engine loaded.')

---
## Part 11: Neural Network Case Predictions

In [ ]:
def predict_case_nn(nos_code, origin_code, class_action, circuit_code, year_filed,
                    has_arbit=False, has_mdl=False, description=''):
    """Predict using the trained neural network."""
    features = encode_case(nos_code, origin_code, class_action, circuit_code,
                           year_filed, has_arbit, has_mdl)
    features = features.reshape(1, -1)
    prob = model.predict(features, verbose=0)[0][0]
    era = get_doctrinal_era(year_filed)
    
    print(f'\n{"=" * 70}')
    if description:
        print(f'  {description}')
    print(f'  NOS: {nos_code} ({NOS_CODE_DESCRIPTIONS.get(nos_code, "")})')
    print(f'  Origin: {origin_code} ({ORIGIN_CODE_LABELS.get(origin_code, "")})')
    print(f'  Circuit: {circuit_code} ({CIRCUIT_CODE_LABELS.get(circuit_code, "")})')
    print(f'  Class Action: {class_action} | Arbitration: {has_arbit} | MDL: {has_mdl}')
    print(f'  Year: {year_filed} | Era: {ERA_NAMES[era]}')
    print(f'{"─" * 70}')
    bar = '█' * int(prob * 50)
    empty = '░' * (50 - int(prob * 50))
    print(f'  NN Plaintiff-Favorable: {prob:6.1%} {bar}{empty}')
    print(f'  Prediction: {"PLAINTIFF-FAVORABLE" if prob >= 0.5 else "NOT PLAINTIFF-FAVORABLE"}')
    return prob

predict_case_nn('480', '1', True, '2', 2020, False, False,
                'FCRA class action — 2nd Cir. — No arbitration')

predict_case_nn('370', '2', False, '5', 2019, True, False,
                'UDAAP individual — Removed — 5th Cir. — Has arbitration')

predict_case_nn('371', '6', True, '3', 2018, False, True,
                'TILA MDL class action — 3rd Cir.')

predict_case_nn('190', '1', False, '4', 2005, False, False,
                'Banking contract — 4th Cir. — Pre-Concepcion')

predict_case_nn('480', '1', False, '5', 2021, False, False,
                'FDCPA individual — 5th Cir. — Post-TransUnion')

predict_case_nn('480', '1', True, '2', 2020, True, False,
                'FCRA class action — 2nd Cir. — WITH arbitration (Concepcion test)')

---
## Part 12: Weighted Engine Predictions with Full Factor Analysis

In [ ]:
def print_weighted_prediction(result):
    """Pretty-print a weighted engine prediction result."""
    print(f'\n{"═" * 70}')
    print(f'  CASE: {result["caseName"]}')
    print(f'  Risk Assessment: {result["riskAssessment"]}')
    print(f'{"─" * 70}')
    
    pf = result['plaintiffFavorableProbability']
    bar = '█' * int(pf * 50)
    empty = '░' * (50 - int(pf * 50))
    print(f'  Plaintiff-Favorable: {pf:6.1%} {bar}{empty}')
    
    print(f'\n  Outcome Distribution:')
    print(f'    Dismissal:          {result["dismissalProbability"]:5.1%}')
    print(f'    Settlement (DISP 6):{result["settlementProbability"]:5.1%}')
    print(f'    Plaintiff Judgment: {result["plaintiffJudgmentProbability"]:5.1%}')
    print(f'    Defendant Judgment: {result["defendantJudgmentProbability"]:5.1%}')
    
    sr = result['expectedSettlementRange']
    print(f'\n  Expected Settlement: ${sr["low"]:,.0f} — ${sr["high"]:,.0f}')
    
    print(f'\n  Key Factors ({len(result["keyFactors"])} total):')
    for f in result['keyFactors']:
        sign = '+' if f['impact'] >= 0 else ''
        icon = '▲' if f['direction'] == 'favorable' else ('▼' if f['direction'] == 'unfavorable' else '─')
        print(f'    {icon} {f["factor"]}: {sign}{f["impact"]*100:.0f}%')
        print(f'      {f["explanation"][:120]}...' if len(f['explanation']) > 120 else f'      {f["explanation"]}')
    
    if result['strategicNotes']:
        print(f'\n  Strategic Notes:')
        for i, note in enumerate(result['strategicNotes']):
            print(f'    [{i+1}] {note[:150]}...' if len(note) > 150 else f'    [{i+1}] {note}')
    print(f'{"═" * 70}')


r1 = predict_outcome_weighted(
    'Doe v. Wells Fargo (FCRA Class)',
    nos_code='480', origin_code='1', class_action=True, circuit_code='2',
    year_filed=2020, has_arbitration=False, concrete_injury=True,
    is_mdl=False, prior_regulatory=False, defendant_bank='Wells Fargo'
)
print_weighted_prediction(r1)

r2 = predict_outcome_weighted(
    'Smith v. Capital One (UDAAP + Arb)',
    nos_code='370', origin_code='2', class_action=False, circuit_code='5',
    year_filed=2022, has_arbitration=True, concrete_injury=False,
    is_mdl=False, prior_regulatory=False, defendant_bank='Capital One'
)
print_weighted_prediction(r2)

r3 = predict_outcome_weighted(
    'Jones v. CashCall (UDAAP + Regulatory)',
    nos_code='370', origin_code='1', class_action=True, circuit_code='9',
    year_filed=2018, has_arbitration=False, concrete_injury=True,
    is_mdl=True, prior_regulatory=True, defendant_bank='CashCall / Western Sky'
)
print_weighted_prediction(r3)

---
## Part 13: Dual-Model Comparison

Compare predictions from both the neural network and weighted engine side by side.

In [ ]:
test_cases = [
    ('FCRA Class — 2nd Cir — No Arb', '480', '1', True, '2', 2020, False, False, True, False, 'Wells Fargo'),
    ('UDAAP Indiv — Removed — 5th Cir — Arb', '370', '2', False, '5', 2022, True, False, False, False, 'Capital One'),
    ('TILA MDL Class — 3rd Cir', '371', '6', True, '3', 2018, False, True, True, False, 'JPMorgan Chase'),
    ('Banking — 4th Cir — Pre-Concepcion', '190', '1', False, '4', 2005, False, False, True, False, 'Other'),
    ('FDCPA — 5th Cir — Post-TransUnion', '480', '1', False, '5', 2021, False, False, True, False, 'Citibank'),
    ('CashCall UDAAP — 9th Cir — Regulatory', '370', '1', True, '9', 2018, False, True, True, True, 'CashCall / Western Sky'),
    ('FCRA + Arb — 2nd Cir (Concepcion)', '480', '1', True, '2', 2020, True, False, True, False, 'Bank of America'),
    ('TILA Indiv — 9th Cir — No Standing', '371', '1', False, '9', 2022, False, False, False, False, 'Wells Fargo'),
]

print(f'{"Case Description":<45} {"NN Prob":>10} {"Weighted":>10} {"Δ":>8} {"Risk":>12}')
print('─' * 90)

for desc, nos, orig, ca, circ, yr, arb, mdl, injury, reg, bank in test_cases:
    nn_features = encode_case(nos, orig, ca, circ, yr, arb, mdl)
    nn_prob = model.predict(nn_features.reshape(1, -1), verbose=0)[0][0]
    
    w_result = predict_outcome_weighted(
        desc, nos_code=nos, origin_code=orig, class_action=ca, circuit_code=circ,
        year_filed=yr, has_arbitration=arb, concrete_injury=injury,
        is_mdl=mdl, prior_regulatory=reg, defendant_bank=bank
    )
    w_prob = w_result['plaintiffFavorableProbability']
    delta = nn_prob - w_prob
    risk = w_result['riskAssessment']
    
    print(f'{desc:<45} {nn_prob:>9.1%} {w_prob:>9.1%} {delta:>+7.1%} {risk:>12}')

### Dual-Model Comparison Visualization

In [ ]:
if 'model' not in dir() or 'predict_outcome_weighted' not in dir():
    raise RuntimeError('Run Parts 7-8 (NN training) and Part 10 (weighted engine) first.')

test_cases_vis = [
    ('FCRA Class\n2nd Cir', '480', '1', True, '2', 2020, False, False, True, False, 'Wells Fargo'),
    ('UDAAP + Arb\n5th Cir', '370', '2', False, '5', 2022, True, False, False, False, 'Capital One'),
    ('TILA MDL\n3rd Cir', '371', '6', True, '3', 2018, False, True, True, False, 'JPMorgan Chase'),
    ('Banking\nPre-Concepcion', '190', '1', False, '4', 2005, False, False, True, False, 'Other'),
    ('FDCPA\nPost-TransUnion', '480', '1', False, '5', 2021, False, False, True, False, 'Citibank'),
    ('CashCall\nUDAAP + Reg', '370', '1', True, '9', 2018, False, True, True, True, 'CashCall / Western Sky'),
    ('FCRA + Arb\nConcepcion', '480', '1', True, '2', 2020, True, False, True, False, 'Bank of America'),
    ('TILA No Standing\n9th Cir', '371', '1', False, '9', 2022, False, False, False, False, 'Wells Fargo'),
]

nn_probs = []
w_probs = []
case_labels = []
risk_labels = []

for desc, nos, orig, ca, circ, yr, arb, mdl, injury, reg, bank in test_cases_vis:
    features = encode_case(nos, orig, ca, circ, yr, arb, mdl)
    nn_p = float(model.predict(features.reshape(1, -1), verbose=0)[0][0])
    w_r = predict_outcome_weighted(
        desc, nos_code=nos, origin_code=orig, class_action=ca, circuit_code=circ,
        year_filed=yr, has_arbitration=arb, concrete_injury=injury,
        is_mdl=mdl, prior_regulatory=reg, defendant_bank=bank
    )
    nn_probs.append(nn_p * 100)
    w_probs.append(w_r['plaintiffFavorableProbability'] * 100)
    case_labels.append(desc)
    risk_labels.append(w_r['riskAssessment'])

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# 1. Side-by-side bar chart
ax = axes[0]
y_pos = np.arange(len(case_labels))
bar_h = 0.35
bars1 = ax.barh(y_pos - bar_h/2, nn_probs, bar_h, label='Neural Network', color='#3498db', alpha=0.85, edgecolor='white')
bars2 = ax.barh(y_pos + bar_h/2, w_probs, bar_h, label='Weighted Engine', color='#e74c3c', alpha=0.85, edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels(case_labels, fontsize=9)
for i, (nn, wp, risk) in enumerate(zip(nn_probs, w_probs, risk_labels)):
    ax.text(max(nn, wp) + 2, i, risk, va='center', fontsize=8, fontweight='bold',
            color={'Very High': '#27ae60', 'High': '#2ecc71', 'Moderate': '#f39c12', 'Low': '#e67e22', 'Very Low': '#e74c3c'}.get(risk, 'gray'))
ax.axvline(x=50, color='#2c3e50', linestyle='--', alpha=0.5, linewidth=1)
ax.set_xlabel('Plaintiff-Favorable Probability (%)')
ax.set_title('Dual-Model Predictions Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.invert_yaxis()
ax.set_xlim(0, 100)

# 2. Scatter plot NN vs Weighted
ax = axes[1]
risk_colors = {'Very High': '#27ae60', 'High': '#2ecc71', 'Moderate': '#f39c12', 'Low': '#e67e22', 'Very Low': '#e74c3c'}
for nn, wp, risk, label in zip(nn_probs, w_probs, risk_labels, case_labels):
    ax.scatter(wp, nn, c=risk_colors.get(risk, 'gray'), s=150, edgecolors='white', linewidth=1.5, zorder=3)
    short_label = label.split('\n')[0]
    ax.annotate(short_label, (wp, nn), textcoords='offset points', xytext=(8, 5), fontsize=7)
ax.plot([0, 100], [0, 100], 'k--', alpha=0.3, linewidth=1, label='Perfect Agreement')
ax.set_xlabel('Weighted Engine (%)')
ax.set_ylabel('Neural Network (%)')
ax.set_title('Model Agreement: NN vs Weighted', fontweight='bold')
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
for risk_name, color in risk_colors.items():
    ax.scatter([], [], c=color, s=80, label=risk_name)
ax.legend(title='Risk Assessment', loc='upper left', fontsize=8)

plt.suptitle('Neural Network vs. Weighted Prediction Engine', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('dual_model_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Dual-model comparison visualization saved.')

---
## Part 14: Full Dataset Summary Statistics

In [ ]:
print('=== COMPLETE FJC DATASET SUMMARY ===')
print(f'Total cases (filtered): {len(df)}')
print(f'Date range: {df["YEAR"].min()} — {df["YEAR"].max()}')
print(f'Plaintiff-favorable rate: {df["PLAINTIFF_FAVORABLE"].mean()*100:.1f}%')

print(f'\n--- NOS Distribution ---')
for code in CONSUMER_BANKING_NOS_CODES:
    n = (df['NOS_STR'] == code).sum()
    pf = df[df['NOS_STR'] == code]['PLAINTIFF_FAVORABLE'].mean()
    print(f'  {code} {NOS_CODE_DESCRIPTIONS[code]}: {n} cases, {pf*100:.1f}% plaintiff-favorable')

print(f'\n--- Circuit Distribution ---')
for circ in sorted(df['CIRCUIT_STR'].unique()):
    n = (df['CIRCUIT_STR'] == circ).sum()
    pf = df[df['CIRCUIT_STR'] == circ]['PLAINTIFF_FAVORABLE'].mean()
    label = CIRCUIT_CODE_LABELS.get(circ, f'Circuit {circ}')
    print(f'  {label}: {n} cases, {pf*100:.1f}% plaintiff-favorable')

print(f'\n--- Origin Distribution ---')
for orig in sorted(df['ORIGIN_STR'].unique()):
    n = (df['ORIGIN_STR'] == orig).sum()
    pf = df[df['ORIGIN_STR'] == orig]['PLAINTIFF_FAVORABLE'].mean()
    label = ORIGIN_CODE_LABELS.get(orig, f'Origin {orig}')
    print(f'  {orig} {label}: {n} cases, {pf*100:.1f}% plaintiff-favorable')

print(f'\n--- DISP Code Distribution ---')
for disp in sorted(df['DISP'].unique()):
    n = (df['DISP'] == disp).sum()
    d_label = DISPOSITION_CODE_LABELS.get(str(int(disp)), '')
    pf_marker = ' *** PLAINTIFF-FAVORABLE' if disp in PLAINTIFF_FAVORABLE_DISP else ''
    print(f'  DISP {int(disp):2d} ({d_label}): {n:5d}{pf_marker}')

print(f'\n--- Doctrinal Era × Plaintiff-Favorable Rate ---')
for i, name in enumerate(ERA_NAMES):
    mask = df['ERA'] == i
    n = mask.sum()
    if n > 0:
        pf = df.loc[mask, 'PLAINTIFF_FAVORABLE'].mean()
        arb = df.loc[mask, 'ARBIT_BIN'].mean()
        ca = df.loc[mask, 'CLASSACT_BIN'].mean()
        print(f'  {name}: {n} cases, {pf*100:.1f}% PF, {arb*100:.1f}% arbitration, {ca*100:.1f}% class action')

---
## Part 15: Save Model & Export

In [ ]:
model.save('fjc_litigation_model.keras')
print('Model saved to fjc_litigation_model.keras')

metadata = {
    'model_name': 'FJC Civil Litigation Binary Outcome Predictor',
    'version': '3.0-binary-fjc-complete',
    'input_dim': INPUT_DIM,
    'output': 'binary sigmoid (plaintiff-favorable)',
    'architecture': '32 → 128 → 64 → 32 → 1',
    'dataset': {
        'source': 'FJC IDB (Federal Judicial Center Integrated Database)',
        'total_rows': len(df),
        'nos_codes': CONSUMER_BANKING_NOS_CODES,
        'plaintiff_favorable_rate': float(df['PLAINTIFF_FAVORABLE'].mean()),
        'year_range': [int(df['YEAR'].min()), int(df['YEAR'].max())],
    },
    'predictor_variables': [
        'NOS (4 one-hot)', 'ORIGIN (8 one-hot)', 'CIRCUIT (12 one-hot)',
        'CLASSACT (binary)', 'ARBIT (binary)', 'MDLDOCK (binary)',
        'YEAR (normalized)', 'ERA (4 one-hot)'
    ],
    'binary_outcome': {
        'positive_class': 'plaintiff-favorable',
        'disp_codes': [1, 6, 7],
        'labels': {'1': 'Judgment for Plaintiff', '6': 'Settled', '7': 'Judgment on Consent'},
    },
    'seminal_cases': [c['citation'] for c in SEMINAL_CASES.values()],
    'weighted_engine_variables': [
        'All 8 FJC variables',
        'Concrete Injury Documented (Spokeo/TransUnion)',
        'Prior Regulatory Action (CashCall/CFPB)',
        'Defendant Bank (institutional risk profile)',
    ],
    'created': datetime.now().isoformat(),
}

with open('fjc_model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('\nMetadata saved to fjc_model_metadata.json')
print(f'\n=== MODEL EXPORT COMPLETE ===')
print(json.dumps(metadata, indent=2))

---
## Part 16: Interactive Prediction (Run Your Own Case)

Modify the parameters below and re-run this cell to predict any case.

In [ ]:
CASE_NAME = 'Your Case v. Defendant Bank'
NOS = '480'          # '480', '190', '371', '370'
ORIGIN = '1'         # '1'-'8'
CLASS_ACTION = False  # True/False
CIRCUIT = '9'        # '0' (DC), '1'-'11'
YEAR = 2024          # Filing year
ARBITRATION = False   # True/False (FJC ARBIT)
MDL = False           # True/False (FJC MDLDOCK)
CONCRETE_INJURY = True  # True/False (Spokeo/TransUnion)
REGULATORY = False    # True/False (CashCall/CFPB)
DEFENDANT = 'Wells Fargo'  # See MAJOR_BANKS list

print('╔══════════════════════════════════════════════════════════════════╗')
print('║           DUAL-MODEL CASE PREDICTION                           ║')
print('╚══════════════════════════════════════════════════════════════════╝')

nn_prob = predict_case_nn(NOS, ORIGIN, CLASS_ACTION, CIRCUIT, YEAR, ARBITRATION, MDL, CASE_NAME)

weighted = predict_outcome_weighted(
    CASE_NAME, nos_code=NOS, origin_code=ORIGIN, class_action=CLASS_ACTION,
    circuit_code=CIRCUIT, year_filed=YEAR, has_arbitration=ARBITRATION,
    concrete_injury=CONCRETE_INJURY, is_mdl=MDL, prior_regulatory=REGULATORY,
    defendant_bank=DEFENDANT
)
print_weighted_prediction(weighted)

print(f'\n{"═" * 70}')
print(f'  SUMMARY')
print(f'  Neural Network:    {nn_prob:6.1%} plaintiff-favorable')
print(f'  Weighted Engine:   {weighted["plaintiffFavorableProbability"]:6.1%} plaintiff-favorable')
print(f'  Risk Assessment:   {weighted["riskAssessment"]}')
print(f'  Settlement Range:  ${weighted["expectedSettlementRange"]["low"]:,.0f} — ${weighted["expectedSettlementRange"]["high"]:,.0f}')
print(f'{"═" * 70}')